In [1]:
import xml.etree.ElementTree as ET
import re

def create_interactive_html(aligned_sentences_xml, output_filename="interactive_text.html"):
    """
    Parses XML-aligned sentences and generates a self-contained HTML file
    with interactive highlighting.

    Args:
        aligned_sentences_xml (list): A list of strings, where each string is an
                                      XML <p> element containing aligned sentences.
        output_filename (str): The name of the HTML file to be created.
    """

    # --- HTML Template ---
    # This template includes all necessary CSS and JavaScript.
    html_template = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Interactive Parallel Text</title>
    <style>
        body {{
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif;
            line-height: 1.8;
            padding: 2em;
            max-width: 900px;
            margin: auto;
            color: #333;
            background-color: #fdfdfd;
        }}
        h1 {{
            border-bottom: 2px solid #e0e0e0;
            padding-bottom: 10px;
            color: #444;
        }}
        .sentence-pair {{
            margin-bottom: 1em;
            padding: 1.5em;
            border-radius: 8px;
            background-color: #f9f9f9;
            border: 1px solid #eee;
        }}
        .sentence-container {{
            margin-bottom: 2.5em;
        }}
        .sentence-pair h2 {{
            margin-top: 0;
            font-size: 1.2em;
            color: #666;
            font-weight: 600;
        }}
        .sentence-pair p {{
            font-size: 1.2em;
            margin-bottom: 0;
        }}
        .greek-text {{
            font-family: "Times New Roman", Times, serif;
            font-size: 1.3em;
        }}
        /* Highlighting class */
        .highlight {{
            background-color: #ffecb3; /* A soft, pleasant yellow */
            cursor: pointer;
            transition: background-color 0.2s ease-in-out;
            border-radius: 3px;
        }}
        span[class^="chunk"] {{
            transition: background-color 0.2s ease-in-out;
        }}
    </style>
    <script>
        // Function to add the 'highlight' class to all elements of a given chunk
        function highlight(chunkClass) {{
            const elements = document.querySelectorAll('.' + chunkClass);
            elements.forEach(el => {{
                el.classList.add('highlight');
            }});
        }}

        // Function to remove the 'highlight' class from all elements of a given chunk
        function unhighlight(chunkClass) {{
            const elements = document.querySelectorAll('.' + chunkClass);
            elements.forEach(el => {{
                el.classList.remove('highlight');
            }});
        }}
    </script>
</head>
<body>
    <h1>Interactive Translation Alignment</h1>
    <p>Mouse over a phrase in either Greek or English to see its corresponding part highlighted in the other language.</p>
    {content}
</body>
</html>
"""

    all_content = ""
    sentence_counter = 0

    # Process each XML sentence string provided in the input list
    for xml_string in aligned_sentences_xml:
        sentence_counter += 1
        # Clean up the XML string to ensure it's a single, well-formed element
        clean_xml = "<p>" + " ".join(xml_string.strip().split()) + "</p>"
        
        try:
            root = ET.fromstring(clean_xml)
        except ET.ParseError as e:
            print(f"Error parsing XML for sentence {sentence_counter}: {e}")
            print(f"Problematic XML: {clean_xml}")
            continue

        greek_parts = []
        english_parts = []
        
        # Iterate through the <foreign> tags to build the HTML spans
        for i, foreign_tag in enumerate(root.findall('.//foreign')):
            chunk_id = f"s{sentence_counter}-chunk{i+1}"
            
            # Extract Greek text (text before the <gloss> tag)
            greek_text = foreign_tag.text.strip() if foreign_tag.text else ""
            
            # Extract English text from the <gloss> tag
            gloss_tag = foreign_tag.find('gloss')
            english_text = gloss_tag.text.strip() if gloss_tag is not None and gloss_tag.text else ""
            
            # Create the HTML spans with the necessary event handlers and class
            greek_span = f'<span class="{chunk_id}" onmouseover="highlight(\'{chunk_id}\')" onmouseout="unhighlight(\'{chunk_id}\')">{greek_text}</span>'
            english_span = f'<span class="{chunk_id}" onmouseover="highlight(\'{chunk_id}\')" onmouseout="unhighlight(\'{chunk_id}\')">{english_text}</span>'
            
            greek_parts.append(greek_span)
            english_parts.append(english_span)

        # Join the parts to form the full sentences
        greek_sentence_html = " ".join(greek_parts)
        english_sentence_html = " ".join(english_parts)

        # Add the complete sentence pair to the overall content
        all_content += f"""
    <div class="sentence-container">
        <div class="sentence-pair">
            <h2>Greek</h2>
            <p class="greek-text">{greek_sentence_html}</p>
        </div>
        <div class="sentence-pair">
            <h2>English</h2>
            <p>{english_sentence_html}</p>
        </div>
    </div>
"""

    # Inject the generated content into the HTML template
    final_html = html_template.format(content=all_content)

    # Write the final HTML to a file
    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write(final_html)
        
    print(f"Successfully created HTML file: {output_filename}")


# --- Main Execution ---
if __name__ == "__main__":
    # A list containing all three aligned sentences in XML format.
    aligned_data = [
        """
        <foreign xml:lang="grc">ἔφη δὲ δεῖν , <gloss>He said that it was the duty of men</gloss></foreign>
        <foreign xml:lang="grc">ἃ μὲν μαθόντας ποιεῖν ἔδωκαν οἱ θεοί , <gloss>whatever the gods had enabled them to do by learning,</gloss></foreign>
        <foreign xml:lang="grc">μανθάνειν , <gloss>to learn</gloss></foreign>
        <foreign xml:lang="grc">ἃ δὲ μὴ δῆλα τοῖς ἀνθρώποις ἐστί , <gloss>whatever was obscure to men ;</gloss></foreign>
        <foreign xml:lang="grc">πειρᾶσθαι <gloss>and to try</gloss></foreign>
        <foreign xml:lang="grc">διὰ μαντικῆς <gloss>by augury</gloss></foreign>
        <foreign xml:lang="grc">παρὰ τῶν θεῶν <gloss>from the gods</gloss></foreign>
        <foreign xml:lang="grc">πυνθάνεσθαι · <gloss>to ascertain</gloss></foreign>
        """,
        """
        <foreign xml:lang="grc">τοὺς θεοὺς γὰρ <gloss>as the gods always</gloss></foreign>
        <foreign xml:lang="grc">οἷς ἂν ὦσιν ἵλεῳ <gloss>to those to whom they are rendered propitious.</gloss></foreign>
        <foreign xml:lang="grc">σημαίνειν . <gloss>afford information</gloss></foreign>
        """,
        """
        <foreign xml:lang="grc">ἀλλὰ μὴν ἐκεῖνός γε <gloss>He was</gloss></foreign>
        <foreign xml:lang="grc">ἀεὶ μὲν ἦν <gloss>constantly</gloss></foreign>
        <foreign xml:lang="grc">ἐν τῷ φανερῷ · <gloss>in public, ;</gloss></foreign>
        """
    ]

    create_interactive_html(aligned_data)


Successfully created HTML file: interactive_text.html
